# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides users through loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport matplotlib.pyplot as plt
# Define the dataset Croissant schema URLurl = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'
# Load the dataset metadatadataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset comprises multiple record sets covering clinical features, hormone levels, and genetic variants.
Let's inspect which record sets are available:

In [ ]:
# List all record sets using their @idrecord_sets = [rs['@id'] for rs in metadata.recordSet]print("Record sets in the dataset:")for rs in metadata.recordSet:    print(f"- {rs['@id']}: {rs.get('name', '[no name]')}")
# For each record set, list fields and columns @idfor rs in metadata.recordSet:    print(f"\nRecordSet: {rs['@id']} (Name: {rs.get('name', '[no name]')})")    fields = rs.get('field', [])    if not fields:
        print("  No fields defined.")        continue    for field in fields:        print(f"  Field @id: {field['@id']}, Name: {field.get('name', '[no name]')}")        columns = field.get('column', [])        if columns:            for column in columns:                print(f"    Column @id: {column['@id']}, Name: {column.get('name', '[no name]')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing entities by their `@id` fields. We'll use the discovered `@id`s above.

Let's load each record set separately so you can visualize and manipulate them.

In [ ]:
# Extract dataframes for each record set using their @idrecord_set_ids = [rs['@id'] for rs in metadata.recordSet]dataframes = {}for record_set_id in record_set_ids:    print(f"\nExtracting records for RecordSet: {record_set_id}")    try:        records = list(dataset.records(record_set=record_set_id))        df = pd.DataFrame(records)        dataframes[record_set_id] = df        print("Columns:", df.columns.tolist())        print(df.head())    except Exception as e:        print(f"Failed to extract records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping.
We'll select a numeric field (e.g., patient age or hormone level) and a grouping field (e.g., phenotype or zygosity) using `@id`s.

In [ ]:
# Example: Choose a record set for clinical features (usually Table 1)# (You may need to adjust these @id based on your earlier exploration)clinical_rs_id = record_set_ids[0] if record_set_ids else None  # Replace with actual Table 1 @id if knowndf = dataframes.get(clinical_rs_id, pd.DataFrame())
# Let's print columns and choose a numeric field and group fieldprint("Available columns:", df.columns.tolist())
# Attempt to find an age column @id (replace with correct @id as needed)age_field_id = Nonefor col in df.columns:    if 'age' in col.lower():        age_field_id = col        breakif age_field_id is None:    age_field_id = df.columns[0] if len(df.columns) > 0 else None
print(f"Using numeric field @id: {age_field_id}")
# Grouping field: try phenotype, diagnosis, or similargroup_field_id = Nonefor col in df.columns:    if 'phenotype' in col.lower() or 'diagnosis' in col.lower():        group_field_id = col        break
# If group field not found, pick clitomegaly or hirsutism as an exampleif group_field_id is None:    for col in df.columns:        if 'clitomegaly' in col.lower() or 'hirsutism' in col.lower():            group_field_id = col            break
print(f"Using group field @id: {group_field_id}")
threshold = 10if age_field_id in df.columns and not df.empty:    # Remove non-numeric values if present    df[age_field_id] = pd.to_numeric(df[age_field_id], errors='coerce')    filtered_df = df[df[age_field_id] > threshold]    print(f"Filtered records with {age_field_id} > {threshold}:")    print(filtered_df.head())
    # Normalization    if not filtered_df.empty:        filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()        print(f"Normalized {age_field_id} for filtered records:")        print(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())
        # Group by group_field_id        if group_field_id and group_field_id in filtered_df.columns:            grouped_df = filtered_df.groupby(group_field_id)[age_field_id].mean().to_frame('mean_' + age_field_id)            print(f"Grouped mean age by {group_field_id}:\n", grouped_df.head())
else:    print("No numeric field for EDA or DataFrame is empty.")

## 5. Visualization
Visualize the age distributions and relationships between key clinical features (example shown for age vs. phenotype, using `@id`).

In [ ]:
if age_field_id and group_field_id and not df.empty:    plt.figure(figsize=(6,4))    if group_field_id in df.columns:        for grp in df[group_field_id].unique():            subset = df[df[group_field_id]==grp]            plt.hist(subset[age_field_id].dropna(), label=str(grp), bins=5, alpha=0.6)        plt.xlabel('Age at Diagnosis')        plt.ylabel('Count')        plt.title('Age Distribution by '+group_field_id)        plt.legend()        plt.show()    else:        # Plot general histogram if group is not available        df[age_field_id].dropna().hist(bins=5)        plt.xlabel('Age at Diagnosis')        plt.ylabel('Count')        plt.title('Age Distribution')        plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, extraction, and basic analysis of the FAIR^2 dataset using `mlcroissant`.

Key findings:
- The dataset is highly granular with record sets describing clinical features, hormone profiles, and genotypes.
- For each entity, all references are via their `@id` for consistency and traceability.
- Exploratory analysis shows patient age distribution and relationships with key features such as phenotype or other diagnosis indicators.

Further steps may include examining hormone levels and genetic variant effects across fields, and expanding visualizations with more record sets or columns.

Refer to Croissant documentation and this notebook template for analyzing other datasets from the schema!